# Notebook 9 - Bowler Season Stats

**Goal**: Pre-compute all bowler statistics used by the Bowlers dashboard page.

**Two outputs**:
- `bowler_phase_season.csv` - one row per bowler per season with powerplay, death, and overall stats
- `bowler_wicket_types.csv` - one row per bowler per season per wicket category for the wicket type breakdown chart

Columns for `bowler_phase_season.csv`:
- `season, bowler` - identifiers
- `pp_balls, pp_runs, pp_wickets, pp_economy` - powerplay
- `death_balls, death_runs, death_wickets, death_economy` - death overs
- `total_balls, total_runs, total_wickets, total_economy` - overall

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

## 2. Load Data

For bowling stats I need two views of the data:
- Legal deliveries only (no wides) for ball counts and wickets
- All deliveries (including wides) for runs conceded - wides are charged to the bowler's economy

In [2]:
df_all   = pd.read_csv('../data/processed/deliveries.csv')
df_all   = df_all[df_all['super_over'] == False].copy()
legal    = df_all[df_all['is_wide'] == False].copy()

print(f'All deliveries (no super overs): {len(df_all):,}')
print(f'Legal deliveries (no wides): {len(legal):,}')
print(f'Seasons: {df_all["season"].min()} - {df_all["season"].max()}')

All deliveries (no super overs): 295,557
Legal deliveries (no wides): 285,686
Seasons: 2008 - 2026


## 3. Phase Stats Per Bowler Per Season

I compute balls bowled (legal), runs conceded (all deliveries), and wickets (excluding run-outs and obstructing the field - those are credited to fielders, not bowlers). Economy = runs conceded / (legal balls / 6).

In [3]:
# Wicket kinds credited to the bowler (excludes run out and obstruction)
BOWLER_WICKETS = ['bowled', 'caught', 'caught and bowled', 'lbw', 'stumped', 'hit wicket']

for phase_name, prefix in [('powerplay', 'pp'), ('death', 'death'), (None, 'total')]:
    if phase_name:
        mask_legal = legal['phase'] == phase_name
        mask_all   = df_all['phase'] == phase_name
    else:
        mask_legal = pd.Series(True, index=legal.index)
        mask_all   = pd.Series(True, index=df_all.index)

    balls_s = (
        legal[mask_legal]
        .groupby(['season', 'bowler'])
        .size()
        .reset_index(name=f'{prefix}_balls')
    )
    runs_s = (
        df_all[mask_all]
        .groupby(['season', 'bowler'])['total_runs']
        .sum()
        .reset_index(name=f'{prefix}_runs')
    )
    wickets_s = (
        legal[mask_legal & legal['wicket'] & legal['wicket_kind'].isin(BOWLER_WICKETS)]
        .groupby(['season', 'bowler'])
        .size()
        .reset_index(name=f'{prefix}_wickets')
    )

    chunk = balls_s.merge(runs_s, on=['season', 'bowler'], how='outer')
    chunk = chunk.merge(wickets_s, on=['season', 'bowler'], how='outer')
    chunk = chunk.fillna(0)
    chunk[f'{prefix}_economy'] = chunk[f'{prefix}_runs'] / (chunk[f'{prefix}_balls'] / 6)

    if prefix == 'pp':
        bowler_wide = chunk
    else:
        bowler_wide = bowler_wide.merge(chunk, on=['season', 'bowler'], how='outer')

bowler_wide = bowler_wide.fillna(0)
print(f'Bowler-season rows: {len(bowler_wide):,}')
print(bowler_wide.head(3))

Bowler-season rows: 2,199
   season    bowler  pp_balls  pp_runs  pp_wickets  pp_economy  death_balls  \
0    2008  A Kumble    24.000   21.000       0.000       5.250       26.000   
1    2008  A Mishra    12.000   21.000       1.000      10.500       18.000   
2    2008   A Nehra   168.000  209.000       5.000       7.464       60.000   

   death_runs  death_wickets  death_economy  total_balls  total_runs  \
0      56.000          1.000         12.923          230         314   
1      20.000          4.000          6.667          120         140   
2      98.000          4.000          9.800          270         357   

   total_wickets  total_economy  
0          7.000          8.191  
1         11.000          7.000  
2         12.000          7.933  


In [4]:
# Preview: top death economy specialists in 2024 (min 50 death balls)
print('Top death economy in 2024 (min 50 death balls):')
print(
    bowler_wide[(bowler_wide['season'] == 2024) & (bowler_wide['death_balls'] >= 50)]
    .sort_values('death_economy')
    .head(10)[['bowler', 'death_balls', 'death_runs', 'death_wickets', 'death_economy']]
    .to_string(index=False)
)

Top death economy in 2024 (min 50 death balls):
        bowler  death_balls  death_runs  death_wickets  death_economy
     JJ Bumrah      101.000     112.000         11.000          6.653
Mohammed Siraj       96.000     140.000          5.000          8.750
  Harshit Rana       85.000     130.000          8.000          9.176
    Avesh Khan      131.000     201.000          8.000          9.206
   M Pathirana       66.000     103.000          5.000          9.364
    N Thushara       62.000     100.000          2.000          9.677
    AD Russell       56.000      92.000          7.000          9.857
Sandeep Sharma       84.000     138.000          8.000          9.857
    Yash Dayal       94.000     155.000          9.000          9.894
     G Coetzee       69.000     114.000          7.000          9.913


## 4. Wicket Type Breakdown

For the wicket type breakdown chart I group wicket kinds into three categories:
- `bowled_lbw`: bowled + lbw (the bowler beat the batter)
- `caught`: caught + caught and bowled + stumped (required a fielder)
- `other`: hit wicket (rare)

This gives a percentage breakdown per bowler per season.

In [5]:
# Only take legal deliveries where the bowler gets credited with the wicket
bowler_wickets = legal[
    legal['wicket'] & legal['wicket_kind'].isin(BOWLER_WICKETS)
][['season', 'bowler', 'wicket_kind']].copy()

# Map to three categories
cat_map = {
    'bowled':           'bowled_lbw',
    'lbw':              'bowled_lbw',
    'caught':           'caught',
    'caught and bowled':'caught',
    'stumped':          'caught',
    'hit wicket':       'other',
}
bowler_wickets['wicket_category'] = bowler_wickets['wicket_kind'].map(cat_map)

wicket_types = (
    bowler_wickets.groupby(['season', 'bowler', 'wicket_category'])
    .size()
    .reset_index(name='count')
)

print(f'Wicket type rows: {len(wicket_types):,}')
print(wicket_types.head(6))

Wicket type rows: 2,963
   season    bowler wicket_category  count
0    2008  A Kumble      bowled_lbw      1
1    2008  A Kumble          caught      6
2    2008  A Mishra      bowled_lbw      3
3    2008  A Mishra          caught      8
4    2008   A Nehra      bowled_lbw      3
5    2008   A Nehra          caught      9


## 5. Save

In [6]:
float_cols = [c for c in bowler_wide.columns if bowler_wide[c].dtype == float]
bowler_wide[float_cols] = bowler_wide[float_cols].round(4)

bowler_wide.to_csv('../data/processed/bowler_phase_season.csv', index=False)
print(f'Saved {len(bowler_wide):,} rows to data/processed/bowler_phase_season.csv')

wicket_types.to_csv('../data/processed/bowler_wicket_types.csv', index=False)
print(f'Saved {len(wicket_types):,} rows to data/processed/bowler_wicket_types.csv')

Saved 2,199 rows to data/processed/bowler_phase_season.csv
Saved 2,963 rows to data/processed/bowler_wicket_types.csv


## Summary

**How the dashboard uses these files**:
- Death economy specialists: filter `death_balls >= 50`, sort by `death_economy`
- Powerplay wicket-takers: filter `pp_balls >= 50`, sort by `pp_wickets`
- Wicket type breakdown: read `bowler_wicket_types.csv`, pivot to wide, compute pct
- PP vs death scatter: merge pp and death qualified subsets
- Season-best (most wickets): filter `total_balls >= 50`, find max `total_wickets` per season

**Saved outputs**:
- `data/processed/bowler_phase_season.csv`
- `data/processed/bowler_wicket_types.csv`